In [1]:
# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# 1. Define exact path and load data
file_path = Path("/home/upw4ys/Documents/floodnet_work/Data_Files/apparently-darling-gecko.parquet")

if not file_path.exists():
    raise FileNotFoundError(f"Target file not found at: {file_path}")

df_gecko = pd.read_parquet(file_path)

# Convert time and downcast to save memory
if 'time' in df_gecko.columns:
    df_gecko['time'] = pd.to_datetime(df_gecko['time'])

for col in df_gecko.select_dtypes(include=['float64']).columns:
    df_gecko[col] = df_gecko[col].astype('float32')

print(f"✅ Data Loaded: {len(df_gecko):,} rows for sensor 'apparently-darling-gecko'.")

✅ Data Loaded: 33,941 rows for sensor 'apparently-darling-gecko'.


In [4]:
df_gecko.columns

Index(['deployment_id', 'time', 'depth_proc_mm', 'depth_inches', 'name',
       'date_deployed', 'date_down', 'deploy_type', 'location', 'image',
       'sensor_mount', 'mounted_over', 'sensor_status', 'coordinates',
       'borough', 'intersection', 'latitude', 'longitude', 'STATION',
       'dist_to_station_ft', 'weather_time', 'elevation [feet]',
       'temp_2m [degF]', 'relative_humidity [percent]', 'dewpoint [degF]',
       'precip_incremental [inch]', 'precip_max_intensity [inch/hour]',
       'precip_1hr [inch]', 'frozen_soil_05cm [bit]', 'frozen_soil_25cm [bit]',
       'frozen_soil_50cm [bit]', 'soil_temp_05cm [degF]',
       'soil_temp_25cm [degF]', 'soil_temp_50cm [degF]',
       'soil_moisture_05cm [m^3/m^3]', 'soil_moisture_25cm [m^3/m^3]',
       'soil_moisture_50cm [m^3/m^3]', 'snow_depth [inch]', 'DATE',
       'minutes_since_weather_update', 'storm_id', 'global_storm_id',
       'storm_start', 'storm_end', 'rain_start', 'rain_end', 'phase',
       'hours_since_rain_st

In [2]:
# %%
# 2. Rain Influence Diagnostics

# Verify required columns exist
required_cols = ['deployment_id', 'total_precip_in', 'net_depth_rise_in', 'precip_1hr [inch]', 'depth_inches']
missing = [c for c in required_cols if c not in df_gecko.columns]
if missing:
    raise ValueError(f"Missing required columns for diagnostics: {missing}")

# Identify storm column dynamically
storm_col = 'global_storm_id' if 'global_storm_id' in df_gecko.columns else ('storm_id' if 'storm_id' in df_gecko.columns else None)
if storm_col is None:
    raise ValueError("Expected either 'global_storm_id' or 'storm_id' in the dataframe.")

# Filter to storm level and drop duplicates
storm_level = (
    df_gecko[['deployment_id', storm_col, 'total_precip_in', 'net_depth_rise_in']]
    .dropna(subset=['deployment_id', storm_col, 'total_precip_in', 'net_depth_rise_in'])
    .drop_duplicates(['deployment_id', storm_col])
)

# Filter for 'wet' events
wet_events = storm_level[storm_level['total_precip_in'] >= 0.05].copy()

rows = []
for did, g in wet_events.groupby('deployment_id', sort=False):
    x = g['total_precip_in'].to_numpy(dtype=float)
    y = g['net_depth_rise_in'].to_numpy(dtype=float)
    
    corr = np.nan
    slope = np.nan
    
    if len(g) >= 2 and np.nanstd(x) > 0 and np.nanstd(y) > 0:
        corr = float(np.corrcoef(x, y)[0, 1])
        slope = float(np.polyfit(x, y, 1)[0])
        
    rows.append({
        'deployment_id': did,
        'n_wet_events': len(g),
        'corr_precip_depth': corr,
        'slope_in_per_in': slope,
        'wet_positive_frac': float((g['net_depth_rise_in'] > 0.02).mean())
    })

rain_influence_summary = pd.DataFrame(rows)
# Calculate dry depth standard deviation (tidal/noise proxy)
dry_conditions = (
    df_gecko[df_gecko['precip_1hr [inch]'] <= 0.001]
    .groupby('deployment_id')['depth_inches']
    .agg(dry_depth_std='std', dry_n='size')
    .reset_index()
)

rain_influence_summary = rain_influence_summary.merge(dry_conditions, on='deployment_id', how='left')

In [3]:
# %%
# 3. Scoring and Normalization

# Normalize metrics (Z-scaling proxy using percentiles)
for c in ['corr_precip_depth', 'slope_in_per_in', 'wet_positive_frac', 'dry_depth_std']:
    vals = rain_influence_summary[c].astype(float)
    lo, hi = np.nanpercentile(vals, 5), np.nanpercentile(vals, 95)
    if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
        rain_influence_summary[c + '_z'] = ((vals - lo) / (hi - lo)).clip(0, 1)
    else:
        rain_influence_summary[c + '_z'] = np.nan

# Handle potential NaNs in penalty fill for single-sensor edge cases
penalty_fill = rain_influence_summary['dry_depth_std_z'].median()
if pd.isna(penalty_fill):
    penalty_fill = 0

# Calculate final Rain Influence Score
rain_influence_summary['rain_influence_score'] = (
    0.45 * rain_influence_summary['corr_precip_depth_z'].fillna(0)
    + 0.35 * rain_influence_summary['slope_in_per_in_z'].fillna(0)
    + 0.20 * rain_influence_summary['wet_positive_frac_z'].fillna(0)
    - 0.25 * rain_influence_summary['dry_depth_std_z'].fillna(penalty_fill)
)

rain_influence_summary = rain_influence_summary.sort_values('rain_influence_score', ascending=False).reset_index(drop=True)

print("\n📊 Diagnostics Complete:")
display(rain_influence_summary)


📊 Diagnostics Complete:


,deployment_id,n_wet_events,corr_precip_depth,slope_in_per_in,wet_positive_frac,dry_depth_std,dry_n,corr_precip_depth_z,slope_in_per_in_z,wet_positive_frac_z,dry_depth_std_z,rain_influence_score
0,apparently-darling-gecko,51,0.264217,0.158758,0.843137,0.608426,20082,NaN,NaN,NaN,NaN,0.0
